# Minería de texto con TextRank

## Contexto práctico

Una empresa recibe reportes extensos de clientes sobre entregas, pagos y soporte. Los responsables necesitan saber rápidamente qué ocurrió, cuáles son los conceptos principales y qué problemas requieren atención.

Leer cada reporte completo toma tiempo. En este ejercicio construiremos una herramienta sencilla que:

1. resume un reporte seleccionando sus oraciones más importantes;
2. extrae palabras y frases clave;
3. permite comparar temas importantes entre varios reportes.

La técnica utilizada será **TextRank**, un algoritmo no supervisado basado en grafos.

## Objetivos de aprendizaje

- Entender TextRank con una explicación sencilla.
- Representar oraciones y palabras como una red.
- Calcular importancia mediante TextRank.
- Generar un resumen extractivo.
- Extraer palabras clave de un reporte.
- Interpretar por qué fueron seleccionadas ciertas oraciones.
- Identificar beneficios y limitaciones para un proceso real.

Los datos están incluidos dentro del notebook. No es necesario subir un archivo externo.

## ¿Qué es TextRank?

TextRank es una técnica de minería de texto basada en grafos. Su nombre puede causar confusión: **no estamos evaluando páginas web ni construyendo un buscador**. TextRank toma un cálculo matemático originalmente conocido como PageRank y lo adapta a texto. En este ejercicio, los nodos son oraciones o palabras, y las conexiones representan similitud o proximidad entre ellas.

La técnica construye un grafo:

- cada oración puede ser un nodo;
- una conexión indica que dos oraciones son similares;
- una puntuación alta indica que una oración está bien conectada con el contenido del documento.

TextRank no genera texto nuevo. Selecciona fragmentos que ya existen, por eso se considera un método extractivo.

## Diferencia frente a otras técnicas

- TF-IDF: mide qué palabras son distintivas.
- LDA: descubre temas en una colección de documentos.
- NER: extrae entidades como personas, lugares y fechas.
- Embeddings: compara el significado entre textos.
- TextRank: ordena oraciones o palabras según su importancia dentro del texto.

En este notebook TextRank se usará para generar un resumen y extraer palabras clave.

## 1. Preparar el entorno

Usaremos NetworkX para construir los grafos y calcular la puntuación TextRank. Internamente, NetworkX utiliza el procedimiento matemático PageRank sobre esos grafos, pero el objeto analizado es el texto. scikit-learn calculará similitud entre oraciones usando TF-IDF; aquí TF-IDF es una representación auxiliar, no el objetivo final del ejercicio.

In [ ]:
%%capture
!pip -q install networkx pandas scikit-learn plotly

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 160)
print('Entorno preparado')

## 2. Dataset incluido en el notebook

Cada registro es un reporte de cliente. El campo tipo es una referencia conocida solo para poder comparar resultados entre grupos; TextRank no utilizará esa columna para seleccionar oraciones ni palabras.

In [ ]:
reportes = [
    {
        'id': 'R001',
        'tipo': 'entrega',
        'texto': 'El cliente reportó que su paquete no llegó en la fecha prometida. El envío aparece detenido desde hace cuatro días en el centro de distribución. La guía indica que el domicilio es correcto y no registra un intento de entrega. El cliente necesita el producto para una instalación programada esta semana. Solicita conocer la nueva fecha de entrega y recibir una explicación por el retraso. El área de logística confirmó que revisará el caso con la empresa de paquetería. También se solicitará una actualización de la guía para informar al cliente.'
    },
    {
        'id': 'R002',
        'tipo': 'facturacion',
        'texto': 'La cliente detectó un cargo duplicado en la factura de este mes. El importe aparece dos veces con la misma referencia de pago. La cuenta bancaria muestra que ambas transacciones fueron procesadas. La cliente solicita la devolución del segundo cargo y un comprobante de la aclaración. El equipo de facturación revisará los movimientos y validará la duplicidad. Si se confirma el error, se iniciará el reembolso correspondiente. La respuesta final será enviada por correo electrónico.'
    },
    {
        'id': 'R003',
        'tipo': 'soporte',
        'texto': 'El usuario no puede iniciar sesión en la aplicación. La pantalla muestra un mensaje de error después de escribir la contraseña. El problema ocurre en dos dispositivos y también aparece al utilizar el portal web. El usuario ya actualizó la aplicación y borró la caché sin obtener una solución. El equipo técnico revisará los registros de autenticación. También se verificará si la cuenta quedó bloqueada por varios intentos. Se enviarán instrucciones para restablecer el acceso.'
    },
    {
        'id': 'R004',
        'tipo': 'producto',
        'texto': 'El cliente recibió un producto diferente al solicitado en su pedido. El modelo entregado no coincide con la descripción mostrada en la compra. El empaque llegó cerrado y no presenta daños visibles. El cliente necesita el modelo correcto para un proyecto que comienza el viernes. Se solicitarán fotografías del producto y del número de pedido. El área de almacén revisará el surtido realizado. Después se coordinará la recolección y el envío del producto correcto.'
    },
    {
        'id': 'R005',
        'tipo': 'cancelacion',
        'texto': 'La cliente solicita cancelar su suscripción mensual. Indica que ya no utiliza el servicio y no desea recibir un nuevo cobro. La cuenta se encuentra activa y no tiene pagos pendientes. El sistema muestra que el siguiente cargo está programado para la próxima semana. Se debe confirmar la identidad antes de procesar la baja. Después de la cancelación se enviará un comprobante al correo registrado. La cliente también solicita información sobre la eliminación de sus datos.'
    }
]

df = pd.DataFrame(reportes)
df.to_csv('reportes_textrank.csv', index=False, encoding='utf-8')
print(f'Reportes cargados: {len(df)}')
display(df[['id', 'tipo', 'texto']])

### ¿Qué beneficio buscamos?

El beneficio no es reemplazar al analista. Es reducir el tiempo de lectura inicial. En lugar de revisar ocho oraciones desde el principio, el responsable puede leer primero tres oraciones relevantes y consultar el documento completo si necesita más contexto.

### Entrada y salida del ejercicio

**Entrada:** un reporte largo formado por varias oraciones.

**Proceso:** TextRank compara las oraciones, construye una red y calcula qué tan conectada está cada una con el resto del reporte.

**Salida:** un resumen formado por oraciones originales y una lista de palabras clave.

**Beneficio:** una persona puede detectar rápidamente el problema principal, los conceptos relevantes y la acción solicitada, sin leer inicialmente todo el documento.

## 3. Preparar el texto

Antes de construir el grafo, separamos cada reporte en oraciones y normalizamos una copia del texto. Conservaremos las oraciones originales para que el resumen sea legible.

In [ ]:
def normalizar_texto(texto):
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = re.sub(r'[^a-z0-9\s]', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

def separar_oraciones(texto):
    oraciones = re.split(r'(?<=[.!?])\s+', str(texto).strip())
    return [oracion.strip() for oracion in oraciones if len(oracion.strip()) > 20]

df['oraciones'] = df['texto'].apply(separar_oraciones)
print(f"Oraciones del primer reporte: {len(df.loc[0, 'oraciones'])}")
display(pd.DataFrame({'oracion': df.loc[0, 'oraciones']}))

### Interpretación de este bloque

La unidad de análisis será la oración. TextRank comparará cada oración contra las demás. Si una oración comparte conceptos con muchas otras, tendrá más conexiones y podrá recibir una puntuación alta.

La función de limpieza crea una versión normalizada para calcular similitudes, pero no modifica el texto original que se mostrará al usuario.

## 4. Construir el grafo de oraciones

Convertimos las oraciones en vectores TF-IDF y calculamos la similitud coseno entre cada par. Cada oración es un nodo; dos nodos se conectan cuando comparten contenido.

La diagonal se pone en cero porque una oración no debe considerarse conectada consigo misma.

In [ ]:
palabras_vacias = [
    'el', 'la', 'los', 'las', 'un', 'una', 'de', 'del', 'en', 'a', 'por',
    'para', 'con', 'que', 'y', 'su', 'sus', 'se', 'lo', 'me', 'no'
]

def construir_grafo_oraciones(oraciones):
    textos_normalizados = [normalizar_texto(oracion) for oracion in oraciones]
    vectorizador = TfidfVectorizer(stop_words=palabras_vacias)
    matriz = vectorizador.fit_transform(textos_normalizados)
    similitudes = cosine_similarity(matriz)
    np.fill_diagonal(similitudes, 0)
    grafo = nx.from_numpy_array(similitudes)
    return grafo, similitudes

oraciones_ejemplo = df.loc[0, 'oraciones']
grafo, matriz_similitud = construir_grafo_oraciones(oraciones_ejemplo)
print(f'Nodos del grafo: {grafo.number_of_nodes()}')
print(f'Conexiones con peso mayor que cero: {grafo.number_of_edges()}')
display(pd.DataFrame(matriz_similitud).round(2))

### Visualización de similitud entre oraciones

El mapa de calor permite ver la materia prima de TextRank. Las celdas más oscuras representan pares de oraciones más parecidos. Las filas y columnas O1, O2, O3, etc. corresponden a las posiciones originales del reporte.

In [ ]:
etiquetas_oraciones = [f'O{i + 1}' for i in range(len(oraciones_ejemplo))]
fig = px.imshow(
    matriz_similitud,
    x=etiquetas_oraciones, y=etiquetas_oraciones, text_auto='.2f',
    color_continuous_scale='Blues', zmin=0, zmax=1,
    labels={'x': 'Oración comparada', 'y': 'Oración de referencia', 'color': 'Similitud coseno'}
)
fig.update_layout(title='TextRank: similitud entre oraciones', height=650)
fig.show()

### Interpretación de la matriz de similitud

Cada fila y columna representa una oración. Un valor alto indica que dos oraciones comparten más vocabulario relevante. Esta matriz se convierte en una red: las oraciones con muchas conexiones importantes tienen mayor influencia en el resumen.

La similitud aquí es principalmente léxica, porque se calcula con palabras. TextRank no comprende causalidad ni verifica hechos; ordena contenido según su relación dentro del documento.

### Visualización: mapa de relación entre oraciones

La siguiente gráfica muestra el grafo que utiliza TextRank. Cada círculo es una oración; el tamaño representa su puntuación de importancia. Las líneas muestran relaciones de similitud. Las oraciones grandes y bien conectadas son candidatas relevantes para el resumen.

In [ ]:
posiciones = nx.spring_layout(grafo, seed=RANDOM_STATE, weight='weight')
# TextRank aplica PageRank al grafo de oraciones; aquí se calcula la puntuación TextRank.
puntuaciones = nx.pagerank(grafo, weight='weight')
edge_x, edge_y = [], []
for origen, destino in grafo.edges():
    edge_x += [posiciones[origen][0], posiciones[destino][0], None]
    edge_y += [posiciones[origen][1], posiciones[destino][1], None]
fig_red = go.Figure()
fig_red.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#B7C9D6'), hoverinfo='none'))
fig_red.add_trace(go.Scatter(
    x=[posiciones[i][0] for i in grafo.nodes()], y=[posiciones[i][1] for i in grafo.nodes()],
    mode='markers+text', text=[f'O{i + 1}' for i in grafo.nodes()], textposition='middle center',
    customdata=[[oraciones_ejemplo[i], puntuaciones[i]] for i in grafo.nodes()],
    hovertemplate='<b>%{text}</b><br>%{customdata[0]}<br>Puntuación TextRank: %{customdata[1]:.4f}<extra></extra>',
    marker=dict(size=[25 + 300 * puntuaciones[i] for i in grafo.nodes()], color=[puntuaciones[i] for i in grafo.nodes()], colorscale='Blues', showscale=True, colorbar=dict(title='TextRank'), line=dict(width=1.5, color='#1F4E79'))
))
fig_red.update_layout(title='TextRank: red interactiva de oraciones', height=700, showlegend=False, xaxis=dict(visible=False), yaxis=dict(visible=False), margin=dict(l=20, r=20, t=70, b=20))
fig_red.show()

## 5. Calcular la puntuación TextRank

En esta etapa calculamos TextRank. Para hacerlo, aplicamos el procedimiento matemático PageRank sobre la red de oraciones creada anteriormente. Una oración recibe una puntuación TextRank alta cuando está conectada con otras oraciones relevantes del mismo documento.

La palabra PageRank aparece únicamente porque es la función matemática disponible en NetworkX. El resultado de este notebook es TextRank: importancia de contenido textual, no ranking de páginas web.

El resultado es un diccionario con una puntuación para cada oración. No debemos interpretar la puntuación como porcentaje de verdad o como probabilidad de que el evento haya ocurrido.

In [ ]:
puntuaciones_oraciones = nx.pagerank(grafo, weight='weight')
ranking_oraciones = pd.DataFrame({
    'posicion_original': list(puntuaciones_oraciones.keys()),
    'oracion': [oraciones_ejemplo[i] for i in puntuaciones_oraciones.keys()],
    'puntuacion_textrank': list(puntuaciones_oraciones.values())
}).sort_values('puntuacion_textrank', ascending=False)
ranking_oraciones['etiqueta_grafica'] = ranking_oraciones.apply(
    lambda fila: f"O{int(fila['posicion_original']) + 1}: {fila['oracion'][:75]}...", axis=1
)

display(ranking_oraciones)
fig_ranking = px.bar(ranking_oraciones.sort_values('puntuacion_textrank'), x='puntuacion_textrank', y='etiqueta_grafica', orientation='h', text='puntuacion_textrank', hover_data={'oracion': True, 'posicion_original': True, 'puntuacion_textrank': ':.4f', 'etiqueta_grafica': False}, color='puntuacion_textrank', color_continuous_scale='Blues', labels={'puntuacion_textrank': 'Puntuación TextRank', 'etiqueta_grafica': 'Oración'})
fig_ranking.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_ranking.update_layout(title='TextRank: importancia relativa de cada oración', height=600, coloraxis_showscale=False, margin=dict(l=20, r=100, t=70, b=40))
fig_ranking.show()

### Interpretación del ranking

Las oraciones que aparecen arriba son candidatas a formar parte del resumen. Una oración puede tener alta puntuación porque conecta varias ideas del reporte, no necesariamente porque sea la primera, la más larga o la última.

Si el resultado selecciona frases repetitivas, puede ser necesario reducir conexiones muy débiles o aplicar una regla de diversidad para no incluir tres oraciones que digan prácticamente lo mismo.

### ¿Por qué esta gráfica ayuda?

La tabla indica la puntuación, pero el grafo explica visualmente de dónde proviene: una oración importante está conectada con otras partes del reporte. Así se entiende que TextRank no selecciona al azar ni elige solamente la oración más larga.

## 6. Generar un resumen extractivo

Tomaremos las tres oraciones con mayor puntuación, pero las devolveremos en el orden original para conservar la secuencia del reporte. Así el resumen utiliza texto real y no inventa información.

In [ ]:
def resumir_con_textrank(texto, numero_oraciones=3):
    oraciones = separar_oraciones(texto)
    if len(oraciones) <= numero_oraciones:
        return ' '.join(oraciones)
    grafo, _ = construir_grafo_oraciones(oraciones)
    # TextRank: PageRank aplicado a la red de oraciones del documento.
    puntuaciones = nx.pagerank(grafo, weight='weight')
    mejores_indices = sorted(
        sorted(puntuaciones, key=puntuaciones.get, reverse=True)[:numero_oraciones]
    )
    return ' '.join(oraciones[indice] for indice in mejores_indices)

resumen = resumir_con_textrank(df.loc[0, 'texto'], numero_oraciones=3)
print('REPORTE ORIGINAL:\n')
print(df.loc[0, 'texto'])
print('\nRESUMEN TEXTRANK:\n')
print(resumen)

### Interpretación del resumen

El resumen conserva oraciones del reporte original. En un uso operativo, el responsable puede leer primero este resultado y decidir si necesita abrir el texto completo.

La calidad debe evaluarse con dos preguntas: ¿el resumen conserva el problema central? y ¿conserva la acción o resolución relevante? Un resumen corto puede omitir detalles importantes, por lo que no debe utilizarse como única fuente en decisiones críticas.

### Medir la reducción de lectura

Esta comparación no mide la calidad semántica por sí sola. Mide el beneficio operativo inmediato: cuántas oraciones puede revisar primero el usuario frente al reporte completo. La calidad se confirma leyendo si el resumen conserva el problema y la acción.

In [ ]:
oraciones_originales = len(separar_oraciones(df.loc[0, 'texto']))
oraciones_resumen = len(separar_oraciones(resumen))
reduccion = 1 - (oraciones_resumen / oraciones_originales)
metricas = pd.DataFrame({
    'medida': ['Oraciones originales', 'Oraciones del resumen', 'Reducción de lectura'],
    'valor': [oraciones_originales, oraciones_resumen, f'{reduccion:.0%}']
})
display(metricas)
comparacion = pd.DataFrame({'version': ['Reporte original', 'Resumen TextRank'], 'oraciones': [oraciones_originales, oraciones_resumen]})
fig_longitud = px.bar(comparacion, x='version', y='oraciones', text='oraciones', color='version', color_discrete_map={'Reporte original': '#A5A5A5', 'Resumen TextRank': '#4472C4'}, labels={'version': '', 'oraciones': 'Número de oraciones'})
fig_longitud.update_traces(textposition='outside')
fig_longitud.update_layout(title='Reducción de lectura mediante TextRank', height=450, showlegend=False, yaxis=dict(dtick=1), margin=dict(t=80, b=50))
fig_longitud.show()

## 7. Extraer palabras clave con TextRank

También podemos construir un grafo de palabras. Cada palabra es un nodo y se conecta con las palabras que aparecen cerca de ella. Las palabras mejor conectadas reciben mayor puntuación y se consideran candidatas a palabras clave.

In [ ]:
def extraer_palabras_clave(texto, numero_palabras=10, ventana=2):
    texto_normalizado = normalizar_texto(texto)
    tokens = [
        token for token in texto_normalizado.split()
        if token not in palabras_vacias and len(token) > 3
    ]
    grafo = nx.Graph()
    grafo.add_nodes_from(set(tokens))
    for posicion, palabra in enumerate(tokens):
        for vecino in tokens[posicion + 1: posicion + 1 + ventana]:
            if palabra != vecino:
                if grafo.has_edge(palabra, vecino):
                    grafo[palabra][vecino]['weight'] += 1
                else:
                    grafo.add_edge(palabra, vecino, weight=1)
    # TextRank: PageRank aplicado a la red de palabras del documento.
    puntuaciones = nx.pagerank(grafo, weight='weight')
    return sorted(puntuaciones.items(), key=lambda elemento: elemento[1], reverse=True)[:numero_palabras]

palabras_clave = extraer_palabras_clave(df.loc[0, 'texto'])
palabras_df = pd.DataFrame(palabras_clave, columns=['palabra', 'puntuacion_textrank'])
display(palabras_df)
fig_palabras = px.bar(palabras_df.sort_values('puntuacion_textrank'), x='puntuacion_textrank', y='palabra', orientation='h', text='puntuacion_textrank', color='puntuacion_textrank', color_continuous_scale='Greens', labels={'puntuacion_textrank': 'Puntuación TextRank', 'palabra': 'Palabra'})
fig_palabras.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_palabras.update_layout(title='TextRank: palabras clave del reporte', height=500, coloraxis_showscale=False, margin=dict(l=20, r=100, t=70, b=40))
fig_palabras.show()

### Interpretación de las palabras clave

Las palabras seleccionadas representan conceptos recurrentes o conectados en el reporte. Pueden utilizarse como etiquetas, filtros de búsqueda o insumo para crear un tablero.

Una palabra clave no es necesariamente un tema completo. Por ejemplo, entrega y guía ayudan a orientar la lectura, pero no explican por sí solas la causa del retraso ni la acción requerida.

## 8. Aplicar TextRank a todos los reportes

Ahora generamos un resumen y palabras clave para cada registro. Esto muestra cómo la técnica puede convertirse en una etapa de preparación para un tablero o sistema de consulta.

In [ ]:
df['resumen_textrank'] = df['texto'].apply(lambda texto: resumir_con_textrank(texto, numero_oraciones=2))
df['palabras_clave'] = df['texto'].apply(
    lambda texto: ', '.join(palabra for palabra, _ in extraer_palabras_clave(texto, numero_palabras=6))
)
display(df[['id', 'tipo', 'resumen_textrank', 'palabras_clave']])
df[['id', 'tipo', 'resumen_textrank', 'palabras_clave']].to_csv(
    'resultados_textrank.csv', index=False, encoding='utf-8'
)

### Beneficio operativo de esta tabla

La tabla puede alimentar una vista de seguimiento: el agente ve el resumen, las palabras clave y el reporte original cuando necesita ampliar. También puede permitir búsquedas como “mostrar reportes relacionados con reembolso” o “encontrar casos donde aparezca entrega”.

En un entorno real se podrían añadir fecha, cliente, prioridad, responsable y estado para analizar tendencias.

## 9. Conclusiones generales

1. TextRank es una técnica no supervisada que ordena oraciones o palabras mediante un grafo de relaciones.
2. El resumen extractivo conserva frases originales, por lo que es fácil rastrear de dónde salió la información.
3. Las palabras clave ayudan a buscar, filtrar y etiquetar documentos.
4. La técnica es útil cuando se necesita una primera lectura rápida sin entrenar un modelo supervisado.
5. La similitud basada en palabras puede perder equivalencias semánticas; para eso embeddings pueden ser una alternativa o complemento.
6. Los resultados deben revisarse cuando el documento sea crítico, ambiguo o contenga información sensible.
7. El beneficio debe medirse con tiempo de lectura, utilidad percibida, precisión del resumen y reducción de trabajo manual.

### Conclusión ejecutiva

TextRank puede convertir reportes largos en una vista breve y navegable: resumen, palabras clave y acceso al documento original. Es una técnica sencilla, transparente y de bajo costo para comenzar un proyecto de análisis documental, siempre manteniendo revisión humana para decisiones importantes.